In [2]:
import torch

def debug_3d_mrope(position_ids, head_dim, theta=10000.0):
    print("=== INPUT ===")
    print("position_ids shape:", position_ids.shape)
    print("position_ids[0] = t:", position_ids[0, 0].tolist())
    print("position_ids[1] = h:", position_ids[1, 0].tolist())
    print("position_ids[2] = w:", position_ids[2, 0].tolist())

    inv_freq = 1.0 / (
        theta ** (
            torch.arange(
                0,
                head_dim,
                2,
                dtype=torch.float32,
                device=position_ids.device,
            ) / head_dim
        )
    )

    print("\n=== STEP 1: inv_freq ===")
    print(inv_freq)

    num_freqs = inv_freq.shape[0]

    axis_for_freq = torch.arange(num_freqs, device=position_ids.device) % 3

    print("\n=== STEP 2: axis_for_freq ===")
    print(axis_for_freq)
    print("0=t, 1=h, 2=w")

    pos_per_freq = position_ids[axis_for_freq]
    print("\n=== pos_per_freq: position_ids[axis_for_freq] ===")
    print(pos_per_freq)

    print("\n=== STEP 3: position_ids[axis_for_freq] ===")
    print("shape before permute:", pos_per_freq.shape)

    pos_per_freq = pos_per_freq.permute(1, 2, 0).to(torch.float32)

    print("shape after permute:", pos_per_freq.shape)
    print("pos_per_freq for each token:")
    print(pos_per_freq[0])

    angles = pos_per_freq * inv_freq

    print("\n=== STEP 4: angles = pos_per_freq * inv_freq ===")
    print(angles[0])

    angles_full = torch.cat([angles, angles], dim=-1)

    print("\n=== STEP 5: duplicate angles to full head_dim ===")
    print("angles_full shape:", angles_full.shape)

    cos = angles_full.cos()
    sin = angles_full.sin()

    print("\n=== STEP 6: output ===")
    print("cos shape:", cos.shape)
    print("sin shape:", sin.shape)

    return cos, sin


position_ids = torch.tensor([
    [[0, 0, 0, 0, 1, 1, 1, 1]],  # t
    [[0, 0, 1, 1, 0, 0, 1, 1]],  # h
    [[0, 1, 0, 1, 0, 1, 0, 1]],  # w
])

cos, sin = debug_3d_mrope(position_ids, head_dim=12)

=== INPUT ===
position_ids shape: torch.Size([3, 1, 8])
position_ids[0] = t: [0, 0, 0, 0, 1, 1, 1, 1]
position_ids[1] = h: [0, 0, 1, 1, 0, 0, 1, 1]
position_ids[2] = w: [0, 1, 0, 1, 0, 1, 0, 1]

=== STEP 1: inv_freq ===
tensor([1.0000e+00, 2.1544e-01, 4.6416e-02, 1.0000e-02, 2.1544e-03, 4.6416e-04])

=== STEP 2: axis_for_freq ===
tensor([0, 1, 2, 0, 1, 2])
0=t, 1=h, 2=w

=== pos_per_freq: position_ids[axis_for_freq] ===
tensor([[[0, 0, 0, 0, 1, 1, 1, 1]],

        [[0, 0, 1, 1, 0, 0, 1, 1]],

        [[0, 1, 0, 1, 0, 1, 0, 1]],

        [[0, 0, 0, 0, 1, 1, 1, 1]],

        [[0, 0, 1, 1, 0, 0, 1, 1]],

        [[0, 1, 0, 1, 0, 1, 0, 1]]])

=== STEP 3: position_ids[axis_for_freq] ===
shape before permute: torch.Size([6, 1, 8])
shape after permute: torch.Size([1, 8, 6])
pos_per_freq for each token:
tensor([[0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 1.],
        [0., 1., 0., 0., 1., 0.],
        [0., 1., 1., 0., 1., 1.],
        [1., 0., 0., 1., 0., 0.],
        [1., 0., 1., 1.

In [3]:
import torch

def debug_patch_merge_no_mlp(x, grid_thw, vision_embed_dim, merge=2):
    outs = []
    offset = 0

    for image_idx, (T, H, W) in enumerate(grid_thw.tolist()):
        print(f"\n=== Image {image_idx} ===")
        print("T,H,W:", T, H, W)

        n = T * H * W
        print("n patches:", n)

        chunk = x[offset : offset + n]
        print("flat chunk shape:", chunk.shape)

        chunk = chunk.reshape(T, H, W, vision_embed_dim)
        print("grid chunk shape:", chunk.shape)

        H_eff = (H // merge) * merge
        W_eff = (W // merge) * merge
        print("H_eff, W_eff:", H_eff, W_eff)

        if H_eff != H or W_eff != W:
            chunk = chunk[:, :H_eff, :W_eff, :].contiguous()
            print("cropped chunk shape:", chunk.shape)

        chunk = chunk.reshape(
            T,
            H_eff // merge,
            merge,
            W_eff // merge,
            merge,
            vision_embed_dim,
        )
        print("after split into groups:", chunk.shape)
        print("shape meaning: (T, H_group, inner_h, W_group, inner_w, D)")

        chunk = chunk.permute(0, 1, 3, 2, 4, 5).contiguous()
        print("after permute:", chunk.shape)
        print("shape meaning: (T, H_group, W_group, inner_h, inner_w, D)")

        chunk = chunk.reshape(
            T * (H_eff // merge) * (W_eff // merge),
            merge * merge * vision_embed_dim,
        )
        print("after flatten groups:", chunk.shape)

        print("\nMerged tokens:")
        print(chunk)

        outs.append(chunk)
        offset += n

    return torch.cat(outs, dim=0)


T, H, W, D = 1, 4, 4, 3

patches = []
for h in range(H):
    for w in range(W):
        patches.append([h, w, h * 10 + w])

x = torch.tensor(patches, dtype=torch.float32)
grid_thw = torch.tensor([[1, 4, 4]])

merged = debug_patch_merge_no_mlp(
    x=x,
    grid_thw=grid_thw,
    vision_embed_dim=D,
    merge=2,
)


=== Image 0 ===
T,H,W: 1 4 4
n patches: 16
flat chunk shape: torch.Size([16, 3])
grid chunk shape: torch.Size([1, 4, 4, 3])
H_eff, W_eff: 4 4
after split into groups: torch.Size([1, 2, 2, 2, 2, 3])
shape meaning: (T, H_group, inner_h, W_group, inner_w, D)
after permute: torch.Size([1, 2, 2, 2, 2, 3])
shape meaning: (T, H_group, W_group, inner_h, inner_w, D)
after flatten groups: torch.Size([4, 12])

Merged tokens:
tensor([[ 0.,  0.,  0.,  0.,  1.,  1.,  1.,  0., 10.,  1.,  1., 11.],
        [ 0.,  2.,  2.,  0.,  3.,  3.,  1.,  2., 12.,  1.,  3., 13.],
        [ 2.,  0., 20.,  2.,  1., 21.,  3.,  0., 30.,  3.,  1., 31.],
        [ 2.,  2., 22.,  2.,  3., 23.,  3.,  2., 32.,  3.,  3., 33.]])


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class PatchMerger(nn.Module):
    def __init__(
        self,
        vision_embed_dim: int,
        llm_hidden_size: int,
        spatial_merge_size: int = 2,
        rms_norm_eps: float = 1e-6,
    ):
        super().__init__()

        self.vision_embed_dim = vision_embed_dim
        self.llm_hidden_size = llm_hidden_size
        self.spatial_merge_size = spatial_merge_size
        self.rms_norm_eps = rms_norm_eps

        self.grouped_dim = vision_embed_dim * (spatial_merge_size ** 2)

        self.fc1 = nn.Linear(self.grouped_dim, self.grouped_dim)
        self.fc2 = nn.Linear(self.grouped_dim, llm_hidden_size)

    def forward(self, x: torch.Tensor, grid_thw: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2:
            raise ValueError(f"x must be (total_patches, vision_embed_dim), got {x.shape}")

        if x.shape[-1] != self.vision_embed_dim:
            raise ValueError(
                f"x last dim must be {self.vision_embed_dim}, got {x.shape[-1]}"
            )

        if grid_thw.ndim != 2 or grid_thw.shape[-1] != 3:
            raise ValueError(f"grid_thw must be (num_images, 3), got {grid_thw.shape}")

        x = F.rms_norm(
            x,
            (self.vision_embed_dim,),
            eps=self.rms_norm_eps,
        )

        merge = self.spatial_merge_size
        outs = []
        offset = 0

        for image_idx, (T, H, W) in enumerate(grid_thw.tolist()):
            n = T * H * W

            chunk = x[offset : offset + n]

            if chunk.shape[0] != n:
                raise ValueError(
                    f"Not enough patches for image {image_idx}: "
                    f"expected {n}, got {chunk.shape[0]}"
                )

            chunk = chunk.reshape(T, H, W, self.vision_embed_dim)

            H_eff = (H // merge) * merge
            W_eff = (W // merge) * merge

            if H_eff == 0 or W_eff == 0:
                raise ValueError(
                    f"H and W must be at least merge size. "
                    f"Got H={H}, W={W}, merge={merge}"
                )

            if H_eff != H or W_eff != W:
                chunk = chunk[:, :H_eff, :W_eff, :].contiguous()

            chunk = chunk.reshape(
                T,
                H_eff // merge,
                merge,
                W_eff // merge,
                merge,
                self.vision_embed_dim,
            )

            chunk = chunk.permute(0, 1, 3, 2, 4, 5).contiguous()

            chunk = chunk.reshape(
                T * (H_eff // merge) * (W_eff // merge),
                self.grouped_dim,
            )

            outs.append(chunk)

            # Important:
            # advance by original n, not cropped n,
            # because x contains the full original image chunk.
            offset += n

        if offset != x.shape[0]:
            raise ValueError(
                f"grid_thw describes {offset} patches, but x has {x.shape[0]} patches"
            )

        x = torch.cat(outs, dim=0)

        return self.fc2(F.gelu(self.fc1(x)))

In [5]:
torch.manual_seed(0)

T, H, W, D = 1, 4, 4, 3
llm_hidden = 5
merge = 2

patches = []
for h in range(H):
    for w in range(W):
        patches.append([h, w, h * 10 + w])

x = torch.tensor(patches, dtype=torch.float32)
grid_thw = torch.tensor([[T, H, W]])

merger = PatchMerger(
    vision_embed_dim=D,
    llm_hidden_size=llm_hidden,
    spatial_merge_size=merge,
)

y = merger(x, grid_thw)

print("input x shape: ", x.shape)
print("grid_thw:      ", grid_thw.tolist())
print("grouped_dim:   ", merger.grouped_dim)
print("output y shape:", y.shape)

input x shape:  torch.Size([16, 3])
grid_thw:       [[1, 4, 4]]
grouped_dim:    12
output y shape: torch.Size([4, 5])
